In [ ]:
silver_products = spark.table("silver_products")

display(silver_products.limit(5))

In [ ]:
dim_product = silver_products.select(
    "id",
    "title",
    "category",
    "brand",
    "sku",
    "price",
    "discountPercentage",
    "rating",
    "stock",
    "availabilityStatus"
)

display(dim_product.limit(5))

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window_spec = Window.orderBy("id")

dim_product = dim_product.withColumn(
    "product_key",
    row_number().over(window_spec)
)

display(dim_product.limit(10))

In [ ]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_product")

In [ ]:
silver_users = spark.table("silver_users")

display(silver_users.limit(5))

In [ ]:
dim_customer = silver_users.select(
    "id",
    "firstName",
    "lastName",
    "age",
    "gender",
    "email",
    "phone",
    "username",
    "birthDate"
)

display(dim_customer.limit(5))

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window_spec = Window.orderBy("id")

dim_customer = dim_customer.withColumn(
    "customer_key",
    row_number().over(window_spec)
)

display(dim_customer.limit(10))

In [ ]:
dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customer")

In [ ]:
from pyspark.sql.functions import min, max

silver_cart_items = spark.table("silver_cart_items")

silver_cart_items.select(
    min("silver_processed_at").alias("min_date"),
    max("silver_processed_at").alias("max_date")
).show()

In [ ]:
from pyspark.sql.functions import sequence, explode, to_date, lit

dim_date = (
    spark.range(1)
    .select(
        explode(
            sequence(
                to_date(lit("2026-01-01")),
                to_date(lit("2026-12-31"))
            )
        ).alias("date")
    )
)

display(dim_date.limit(10))

In [ ]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    quarter,
    date_format
)

dim_date = (
    dim_date
    .withColumn("year", year("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day", dayofmonth("date"))
)

display(dim_date.limit(10))

In [ ]:
from pyspark.sql.functions import date_format

dim_date = dim_date.withColumn(
    "date_key",
    date_format("date", "yyyyMMdd").cast("int")
)

display(dim_date.limit(10))

In [ ]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_date")

In [ ]:
silver_cart_items = spark.table("silver_cart_items")

display(silver_cart_items.limit(10))

In [ ]:
from pyspark.sql.functions import col

fact_sales = (
    silver_cart_items
    .join(
        dim_product.select(
            col("id").alias("product_id"),
            col("product_key")
        ),
        on="product_id",
        how="left"
    )
    .join(
        dim_customer.select(
            col("id").alias("user_id"),
            col("customer_key")
        ),
        on="user_id",
        how="left"
    )
)

display(fact_sales.limit(10))

In [ ]:
fact_sales = fact_sales.select(
    "cart_id",
    "product_key",
    "customer_key",
    "quantity",
    "price",
    "gross_amount",
    "discount_amount",
    "discounted_total"
)

display(fact_sales.limit(10))

In [ ]:
from pyspark.sql.functions import to_date, date_format

fact_sales = (
    silver_cart_items
    .join(
        dim_product.select(
            "id",
            "product_key"
        ),
        silver_cart_items.product_id == dim_product.id,
        "left"
    )
    .join(
        dim_customer.select(
            "id",
            "customer_key"
        ),
        silver_cart_items.user_id == dim_customer.id,
        "left"
    )
    .withColumn(
        "date_key",
        date_format(
            to_date("silver_processed_at"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "cart_id",
        "product_key",
        "customer_key",
        "date_key",
        "quantity",
        "price",
        "gross_amount",
        "discount_amount",
        "discounted_total"
    )
)

display(fact_sales.limit(10))

In [ ]:
from pyspark.sql.functions import col, sum

fact_sales.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in fact_sales.columns
    ]
).show()

In [ ]:
print("FactSales rows:", fact_sales.count())

In [ ]:
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_sales")

In [ ]:
for table_name in [
    "dim_product",
    "dim_customer",
    "dim_date",
    "fact_sales"
]:
    print(table_name, "→", spark.table(table_name).count(), "rows")

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

dim_product = (
    silver_products
    .select(
        "id",
        "title",
        "category",
        "brand",
        "sku",
        "price",
        "discountPercentage",
        "rating",
        "stock",
        "availabilityStatus"
    )
    .dropDuplicates(["id"])
)

window_spec = Window.orderBy("id")

dim_product = dim_product.withColumn(
    "product_key",
    row_number().over(window_spec)
)

print("dim_product rows:", dim_product.count())
display(dim_product.limit(10))

In [ ]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_product")

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

dim_customer = (
    silver_users
    .select(
        "id",
        "firstName",
        "lastName",
        "age",
        "gender",
        "email",
        "phone",
        "username",
        "birthDate"
    )
    .dropDuplicates(["id"])
)

window_spec = Window.orderBy("id")

dim_customer = dim_customer.withColumn(
    "customer_key",
    row_number().over(window_spec)
)

print("dim_customer rows:", dim_customer.count())
display(dim_customer.limit(10))

In [ ]:
print("silver_users rows:", silver_users.count())
print("distinct user IDs:", silver_users.select("id").distinct().count())

silver_users.groupBy("id") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20)

In [ ]:
dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customer")

In [ ]:
print(
    "FactSales rows:",
    fact_sales.count()
)

print(
    "Distinct product keys in Fact:",
    fact_sales.select("product_key").distinct().count()
)

print(
    "Distinct customer keys in Fact:",
    fact_sales.select("customer_key").distinct().count()
)

print(
    "Product dimension rows:",
    dim_product.count()
)

print(
    "Customer dimension rows:",
    dim_customer.count()
)

In [ ]:
from pyspark.sql.functions import to_date, date_format

fact_sales = (
    silver_cart_items
    .join(
        dim_product.select("id", "product_key"),
        silver_cart_items.product_id == dim_product.id,
        "left"
    )
    .join(
        dim_customer.select("id", "customer_key"),
        silver_cart_items.user_id == dim_customer.id,
        "left"
    )
    .withColumn(
        "date_key",
        date_format(
            to_date("silver_processed_at"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "cart_id",
        "product_key",
        "customer_key",
        "date_key",
        "quantity",
        "price",
        "gross_amount",
        "discount_amount",
        "discounted_total"
    )
)

print("FactSales rows:", fact_sales.count())
print(
    "Distinct product keys:",
    fact_sales.select("product_key").distinct().count()
)
print(
    "Distinct customer keys:",
    fact_sales.select("customer_key").distinct().count()
)

In [ ]:
print("Silver cart items:", silver_cart_items.count())

print(
    "Distinct product IDs in silver_cart_items:",
    silver_cart_items.select("product_id").distinct().count()
)

print(
    "Distinct product IDs in dim_product:",
    dim_product.select("id").distinct().count()
)

In [ ]:
fact_sales.select(
    "product_key",
    "customer_key"
).filter(
    "product_key IS NULL OR customer_key IS NULL"
).count()

In [ ]:
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_sales")